# INFS3208 Programming Assignment Task II
## Distributed Semantic Search with Kubernetes and FAISS

This notebook is the programming interface for **Objective 3**. You will complete the **embedding**, **FAISS indexing**, and **FAISS query** operations. All distribution-related operations (shard discovery, data partitioning, fan-out query execution, and global Top-K merging) are provided by the teaching team.

### Important constraints
- Do **not** change `doc_id` values.
- Preserve a one-document-to-one-vector mapping.
- You may apply additional text-level preprocessing if you believe it is useful.


## Part 0 - Load the supplied dataset and provided runtime

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd

# Course-provided runtime. Do not modify distributed_runtime.py.
for candidate in [Path('/opt/assignment/runtime'), Path('../provided'), Path('provided')]:
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from distributed_runtime import (
    discover_shards,
    publish_student_code,
    distributed_index,
    distributed_search,
    shard_status,
)

DATA_DIR = Path('.')
if not (DATA_DIR / 'documents.csv').exists():
    DATA_DIR = Path('../data')

documents = pd.read_csv(DATA_DIR / 'documents.csv')
queries = pd.read_csv(DATA_DIR / 'queries.csv')

print(f'Documents: {len(documents):,}')
print(f'Queries: {len(queries):,}')
display(documents.head(3))
display(queries.head(3))


## Part 1 - Optional text preprocessing
Course staff have already performed sentence-aware chunking and conservative whitespace normalisation. You may add other **text-level** preprocessing, but document IDs and row count must not change.

In [ ]:
# Optional: you may revise this function to apply additional text-level preprocessing.
# You MUST preserve the number/order of documents and their doc_id values.
def preprocess_text(text: str) -> str:
    return str(text).strip()

processed_texts = [preprocess_text(text) for text in documents['text'].tolist()]
assert len(processed_texts) == len(documents)


## Part 2 - Generate document embeddings (1.0 mark)
Use the supplied `all-MiniLM-L6-v2` sentence-transformer model to convert every document into a dense vector.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_PATH = '/opt/models/all-MiniLM-L6-v2'

# TODO 1 (0.25 marks): Load the supplied sentence-transformer model.
model = 

# TODO 2 (0.75 marks): Generate one dense embedding for each processed document.
# Requirements:
#   - use processed_texts
#   - return a NumPy array
#   - do not normalise embeddings here; FAISS preprocessing is assessed separately
document_embeddings = 

print('Embedding shape:', document_embeddings.shape)
print('Embedding dtype:', document_embeddings.dtype)
assert document_embeddings.ndim == 2
assert document_embeddings.shape[0] == len(documents)
assert document_embeddings.dtype == np.float32

## Part 3 - Implement FAISS indexing and local search (1.5 marks)
Complete the two functions below. These exact functions will later be distributed to the FAISS shard Pods.

In [ ]:
%%writefile student_faiss.py
import faiss
import numpy as np


def build_index(vectors):
    """Build one local FAISS shard index and return it."""
    # TODO 3 (1 mark): Convert the input vectors to float32 without changing row order.
    index = 
    
    return index


def search_index(index, query_vector, k):
    """Search one local FAISS shard and return (scores, local_indices)."""
    # TODO 4 (1 mark): Run a Top-k FAISS search and return scores and local indices.
    scores, local_indices = 
    
    return scores, local_indices


In [ ]:
# Load your student module and build a local index for validation.
import student_faiss
importlib.reload(student_faiss)

local_index = student_faiss.build_index(document_embeddings)
print('Local index vectors:', local_index.ntotal)
assert local_index.ntotal == len(documents)


## Part 4 - Embed and query one text input (0.5 marks)
Generate the query embedding with the same embedding model, then use your `search_index()` function.

In [ ]:
# Select one supplied natural-language query. You may also replace it with your own query.
query_text = str(queries.iloc[0]['question'])
print('Query:', query_text)

# Apply the same optional text preprocessing used for documents.
processed_query = preprocess_text(query_text)

# TODO 5 (0.5 marks): Generate a float32 embedding for this query using the same model.
query_embedding = 

# Use your assessed search_index() implementation.
TOP_K = 5
scores, local_indices = student_faiss.search_index(local_index, query_embedding, TOP_K)

rows = []
for rank, (score, idx) in enumerate(zip(scores[0], local_indices[0]), start=1):
    if idx < 0:
        continue
    row = documents.iloc[int(idx)]
    rows.append({
        'rank': rank,
        'score': float(score),
        'doc_id': int(row['doc_id']),
        'title': row['title'],
        'text': row['text'],
    })
display(pd.DataFrame(rows))


## Distributed execution (provided; no distribution code is assessed)

The cells below reuse **your** `build_index()` and `search_index()` functions on every reachable FAISS shard. The teaching-team runtime performs shard discovery, modulo data partitioning, network communication, concurrent fan-out, and global Top-K merging.

If you temporarily scaled the StatefulSet to one replica, this code will discover one shard and place all documents on it. Before final validation, restore **three replicas** and rerun distributed indexing.


In [ ]:
shards = discover_shards()
print('Reachable shards:', shards)

# Publish the student implementation to every reachable shard.
publish_result = publish_student_code('student_faiss.py', shards=shards)
publish_result


In [ ]:
# The provided runtime partitions by doc_id % number_of_reachable_shards.
index_summary = distributed_index(
    document_embeddings,
    documents['doc_id'].to_numpy(),
    shards=shards,
)
index_summary


In [ ]:
distributed_result = distributed_search(query_embedding, k=TOP_K, shards=shards)
print('Shards queried:', distributed_result['num_shards'])

result_rows = []
for rank, item in enumerate(distributed_result['global_top_k'], start=1):
    row = documents.loc[documents['doc_id'] == item['doc_id']].iloc[0]
    result_rows.append({
        'rank': rank,
        'score': item['score'],
        'doc_id': item['doc_id'],
        'shard': item['shard'],
        'title': row['title'],
        'text': row['text'],
    })
display(pd.DataFrame(result_rows))

# Final required deployment check. During development this may be 1; for final validation it must be 3.
print('Current shard status:')
shard_status(shards)


### Submission reminder
Download/save this completed notebook as **`FAISS.ipynb`**. Your submission ZIP must also contain the completed `kubernetes.yaml` and `commands.sh`. Do not submit `student_faiss.py` separately; it is generated from this notebook and will be recreated during marking.
